In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.listdir('/content/drive/MyDrive/OULAD_processed')

In [ ]:
import pandas as pd
OUTPUT_PATH = '/content/drive/MyDrive/OULAD_processed/'

# ════════════════════════════════════════════════════════════
# MODULE M5 — PERSONNALISATION
# Couche 1 : Règles métier
# Couche 2 : Filtrage hybride (profil + risk + pairs)
# ════════════════════════════════════════════════════════════

In [ ]:
import pickle
import numpy as np
import pandas as pd

# ── Datasets ─────────────────────────────
final_df = pd.read_csv(
    OUTPUT_PATH + 'oulad_final.csv'
)

final_df_wcdmacp = pd.read_csv(
    OUTPUT_PATH +
    'oulad_final_with_code_module_and_code_presentation.csv'
)

weekly_combined = pd.read_csv(
    OUTPUT_PATH + 'oulad_weekly_v2.csv'
)

# ── Embeddings ───────────────────────────
embeddings_full = np.load(
    OUTPUT_PATH + 'gru_embeddings_full.npy'
)

# ── Risk scores GRU ──────────────────────
with open(
    OUTPUT_PATH + 'gru_checkpoint_risk_scores.pkl',
    'rb'
) as f:
    risk_scores_cp = pickle.load(f)

checkpoint_results = {
    3: {
        'preds': risk_scores_cp[3]
    }
}

# ── Student IDs ──────────────────────────
student_ids = final_df_wcdmacp[
    ['id_student',
     'code_module',
     'code_presentation']
].drop_duplicates().values.tolist()

In [ ]:
with open(OUTPUT_PATH + 'gru_risk_scores_checkpoints.pkl', 'rb') as f:
    risk_scores_cp_all = pickle.load(f)

print(risk_scores_cp_all[3].shape)

In [ ]:
checkpoint_results[3]['preds'] = risk_scores_cp_all[3]

In [ ]:
DYN_PATH = OUTPUT_PATH + "dynamic_profiling_kmeans_fixed_labels3/"

final_df_wcdmacp = pd.read_csv(
    DYN_PATH + "final_df_with_kmeans_profiles.csv"
)

final_with_profiles = final_df_wcdmacp.copy()

In [ ]:
# Charger assessments
PATH = '/content/drive/MyDrive/OULAD/'
assessments_df = pd.read_csv(PATH + "assessments.csv")

In [ ]:
assessments_df['date'] = pd.to_numeric(
    assessments_df['date'],
    errors='coerce'
)

assessments_df = assessments_df.dropna(subset=['date'])

# Semaines 1 à 14
assessments_1_14 = assessments_df[
    assessments_df['date'].between(1, 98)
]

print("Nombre total d'évaluations :", len(assessments_1_14))

print("\nPar type :")
print(
    assessments_1_14['assessment_type']
    .value_counts()
)

In [ ]:
pcea_df = pd.read_csv(
    DYN_PATH + "pcea_kmeans_results.csv"
)

print(pcea_df.shape)
print(pcea_df.columns.tolist())

display(pcea_df.head())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("MODULE M5 — PERSONNALISATION")
print("Couche 1 : Règles métier")
print("Couche 2 : Filtrage hybride")
print("=" * 60)

# ── Constantes ────────────────────────────────────────────────
OPTIMAL_THRESHOLD = 0.35
CHECKPOINTS       = [3, 7, 12]
student_id_cols   = ['id_student', 'code_module',
                     'code_presentation']

# ════════════════════════════════════════════════════════════
# CONFIGURATION — Types de contenu et recommandations
# ════════════════════════════════════════════════════════════
CONTENT_BY_PROFILE = {
    'Resource Skimmer': {
        'content_types' : ['resource', 'url', 'oucontent'],
        'description'   : 'Structured and guided content',
        'rationale'     : 'Low-activity student — simple and accessible content',
    },
    'At-Risk Engager': {
        # Re-engagement via contenu interactif + social
        # url (534) pour contenu externe accessible
        'content_types' : ['oucontent', 'forumng', 'url'],
        'description'   : 'Interactive content + support forums',
        'rationale'     : 'Engagement present but moderate risk —'
                          'stimulate via interactive and social content',
    },
    'Mid-Engager': {
        # Consolidation via quiz + contenu + forum
        # quiz (114) pour tester les connaissances
        'content_types' : ['oucontent', 'quiz', 'forumng'],
        'description'   : 'Performance and consolidation-oriented content',
        'rationale'     : 'High engagement but academic performance'
                          'needs consolidating — focus on quality rather than '
                          'quantity',
    },
    'Assessment Specialist': {
        # Approfondissement via quiz + wiki collaboratif
        # ouwiki (43) pour production de contenu avancé
        'content_types' : ['quiz', 'ouwiki', 'oucontent'],
        'description'   : 'Assessment-oriented and in-depth content',
        'rationale'     : 'Strong on assessments — push toward '
                          'deeper learning and collaboration',
    },
    'Highly Engaged Learner': {
        # Contenu avancé collaboratif
        # pas de quiz → déjà performant académiquement
        'content_types' : ['oucontent', 'ouwiki', 'forumng'],
        'description'   : 'Contenu avancé et collaboratif',
        'rationale'     : 'Engaged student — enriching and collaborative content',
    },
}

def get_difficulty(risk_score):
    if risk_score >= 0.7:
        return 'facile'        # révision et consolidation
    elif risk_score >= OPTIMAL_THRESHOLD:
        return 'moyen'         # renforcement
    else:
        return 'difficile'     # approfondissement

def get_difficulty_description(profile, niveau):
    """
    CORRECTION 2 : description cohérente avec le niveau réel,
    pas seulement avec le profil.
    """
    base = CONTENT_BY_PROFILE.get(
        profile, CONTENT_BY_PROFILE['At-Risk Engager']
    )
    if niveau == 'facile':
        return 'Review content and fundamentals consolidation'
    elif niveau == 'moyen':
        return 'Reinforcement content tailored to the profile'
    else:
        # niveau difficile → garder la description du profil
        return base['description']

def get_rhythm(click_regularity, risk_score=0.5):
    """
    Croise la régularité comportementale et le risk score
    pour recommander un rythme adapté.
    """
    # Étudiant très à risque → sessions courtes peu importe la régularité
    if risk_score >= 0.7:
        return 'Sessions courtes quotidiennes (15-20 min/jour)'

    # Étudiant modérément à risque
    elif risk_score >= 0.35:
        if click_regularity < 10:
            return 'Sessions courtes quotidiennes (15-20 min/jour)'
        else:
            return 'Sessions moyennes 3x/semaine (45 min)'

    # Étudiant non à risque → selon sa régularité naturelle
    else:
        if click_regularity < 5:
            return 'Sessions courtes quotidiennes (15-20 min/jour)'
        elif click_regularity < 20:
            return 'Sessions moyennes 3x/semaine (45 min)'
        else:
            return 'Sessions longues 2x/semaine (1h30)'



In [ ]:
def has_upcoming_evaluation(code_module, code_presentation,
                             semaine_courante, assessments_df,
                             window=2):

    start_day = (semaine_courante - 1) * 7
    end_day   = (semaine_courante + window) * 7

    upcoming = assessments_df[
        (assessments_df['code_module'] == code_module) &
        (assessments_df['code_presentation'] == code_presentation) &
        (assessments_df['date'].between(start_day, end_day))
    ]

    return len(upcoming) > 0

In [ ]:
def has_recent_evaluation(code_module, code_presentation, semaine_courante,
                          assessments_df, window=2):

    start_day = max(0, (semaine_courante - window) * 7)
    end_day   = semaine_courante * 7

    recent = assessments_df[
        (assessments_df['code_module'] == code_module) &
        (assessments_df['code_presentation'] == code_presentation) &
        (assessments_df['date'].between(start_day, end_day))
    ]

    return len(recent) > 0

In [ ]:
# ════════════════════════════════════════════════════════════
# COUCHE 1 — RÈGLES MÉTIER
# Détection des cas évidents → alerte immédiate
# ════════════════════════════════════════════════════════════

PROFILE_COL_BY_CP = {
    3: 'profile_label_w3',
    7: 'profile_label_w7',
    12: 'profile_label_w12',
    38: 'profile_label'
}

def adjust_priority(base_priority, risk_score):
    """
    Ajuste la priorité selon le score de risque.
    Ne diminue jamais la priorité métier.
    """

    priority_rank = {
        'moyenne': 1,
        'haute': 2,
        'critique': 3
    }

    # priorité suggérée par le modèle
    if risk_score >= 0.80:
        score_priority = 'critique'
    elif risk_score >= 0.60:
        score_priority = 'haute'
    else:
        score_priority = 'moyenne'

    # garder la plus élevée
    final_priority = max(
        base_priority,
        score_priority,
        key=lambda p: priority_rank[p]
    )

    return final_priority

def couche1_regles_metier(student_row, weekly_data,semaine_courante, assessments_df):
    """
    Applique les règles métier dans le bon ordre :
    R3 (silence total 2 sem.) → R2 (inactivité 2 sem.)
    → R1 (inactivité cette sem.) → R4 (chute) → R5 (soumission)

    Retourne une alerte si cas évident, None sinon.
    """

    profile_col = PROFILE_COL_BY_CP.get(
        semaine_courante, 'profile_label'
    )


    sid     = student_row['id_student']
    mod     = student_row['code_module']
    pres    = student_row['code_presentation']
    profile = student_row.get(profile_col, student_row.get('profile_label', 'At-Risk Engager'))
    reg     = student_row.get('click_regularity', 10)
    risk_score = student_row.get('risk_score', 0.5)

    # Récupérer le contenu et le rythme selon le profil
    content_config = CONTENT_BY_PROFILE.get(
        profile, CONTENT_BY_PROFILE['At-Risk Engager']
    )
    #rythme = get_rhythm(reg, risk_score=student_row.get('risk_score', 0.5))

    week_ref = semaine_courante - 1

    recent = weekly_data[
        (weekly_data['id_student']        == sid)  &
        (weekly_data['code_module']        == mod)  &
        (weekly_data['code_presentation']  == pres) &
        (weekly_data['week'].between(
            max(0, week_ref - 2), week_ref
        ))
    ].sort_values('week')

    clicks_this = recent[
        recent['week'] == week_ref
    ]['weekly_clicks'].sum()

    clicks_last = recent[
        recent['week'] == week_ref - 1
    ]['weekly_clicks'].sum()

    recent_eval = has_recent_evaluation( code_module=mod, code_presentation=pres,
                  semaine_courante=semaine_courante, assessments_df=assessments_df,
                  window=2 )

    submitted_recent = recent['weekly_n_submitted'].sum()


    # ── Règle 3 : Silence total ───────────────────────────────
    if clicks_this == 0 and clicks_last == 0:

        # Vérifier si évaluation imminente
        eval_imminente = has_upcoming_evaluation(
            code_module       = mod,
            code_presentation = pres,
            semaine_courante  = semaine_courante,
            assessments_df    = assessments_df,
            window            = 2
        )

        if eval_imminente:
            return {
                'rule'             : 'R3',
                'type'             : 'ALERTE_SILENCE_PRE_EVAL',
                'priorite'         : adjust_priority('critique', risk_score),
                'message'          : "No activity recorded in the 2 weeks preceding an assessment",
                'action'           : "Urgent reminder + revision resources",
                'niveau_difficulte': 'facile',
                'difficulte'       : 'facile',
                'content_types'    : content_config['content_types'],
                #'rythme'           : rythme,
                'rationale'        : f"Total silence before assessment — "
                                     f"profile {profile} requires immediate "
                                     f"re-engagement",
                'source'           : 'règles',
            }
        else:
            return {
                'rule'             : 'R3',
                'type'             : 'ALERTE_SILENCE_TOTAL',
                'priorite'         : adjust_priority('critique', risk_score),
                'message'          : "No activity at all recorded over the past 2 weeks",
                'action'           : "Teacher contact + "
                                     "re-engagement resources",
                'niveau_difficulte': 'facile',
                'difficulte'       : 'facile',
                'content_types'    : content_config['content_types'],
                #'rythme'           : rythme,
                'rationale'        : f"Total silence detected — profile "
                                     f"{profile} requires immediate "
                                     f"re-engagement",
                'source'           : 'règles',
            }


    # ── Règle 2 : Inactivité consécutive ─────────────────────
    # Vérifié avant R1 car plus spécifique

    if clicks_this < 5 and clicks_last < 5:
        return {
            'rule'             : 'R2',
            'type'             : 'ALERTE_DECROCHAGE',
            'priorite'         : adjust_priority('critique', risk_score),
            'message'          : "Low activity observed for 2 consecutive weeks",
            'action'           : "Teacher contact + remedial conten",
            'niveau_difficulte': 'facile',
            'difficulte'       : 'facile',
            'content_types'    : content_config['content_types'],
            #'rythme'           : rythme,
            'rationale'        : f"Prolonged inactivity — profile {profile} "
                                 f"requires direct contact",
            'source'           : 'règles',
        }


    # ── Règle 1 : Inactivité totale (cette semaine seulement) ─

    if clicks_this < 5:
        return {
            'rule'             : 'R1',
            'type'             : 'ALERTE_CRITIQUE',
            'priorite'         : adjust_priority('haute', risk_score),
            'message'          : f"Very low activity observed during week {semaine_courante}",
            'action'           : "Review the module's core resources",
            'niveau_difficulte': 'facile',
            'difficulte'       : 'facile',
            'content_types'    : content_config['content_types'],
            #'rythme'           : rythme,
            'rationale'        : f"Inactivity detected in week {semaine_courante} "
                                 f"— profile {profile} needs re-engagement",
            'source'           : 'règles',
        }


    # ── Règle 4 : Chute brutale ≥ 50% ────────────────────────

    if clicks_last > 0:
        chute = (clicks_last - clicks_this) / (clicks_last + 1e-8)
        if chute >= 0.5:
            return {
                'rule'             : 'R4',
                'type'             : 'ALERTE_CHUTE',
                'priorite'         : adjust_priority('moyenne', risk_score),
                'message'          : f"Activity drop of {chute*100:.0f}% compared to the previous week",
                'action'           : "Review quiz + lighter resources",
                'niveau_difficulte': 'moyen',
                'difficulte'       : 'moyen',
                'content_types'    : content_config['content_types'],
                #'rythme'           : rythme,
                'rationale'        : f"Drop of {chute*100:.0f}% — profile {profile} "
                                     f"showing partial disengagement",
                'source'           : 'règles',
            }


    # ── Règle 5 : Aucune soumission ───────────────────────────

    if submitted_recent == 0 and recent_eval and semaine_courante >= 4:
        return {
            'rule'             : 'R5',
            'type'             : 'ALERTE_SOUMISSION',
            'priorite'         : adjust_priority('moyenne', risk_score),
            'message'          : "No submission recorded over the past 2 weeks",
            'action'           : "Reminder of pending assessments",
            'niveau_difficulte': 'moyen',
            'difficulte'       : 'moyen',
            'content_types'    : content_config['content_types'],
            #'rythme'           : rythme,
            'rationale'        : f"No submission detected — profile {profile} "
                                 f"needs a deadline reminder",
            'source'           : 'règles',
        }

    return None  # Pas de cas évident → couche 2


In [ ]:
FEAT_COLS_BY_CP = {
    3: [
        # Activité globale
        'clicks_sum_w3', 'clicks_mean_w3', 'clicks_std_w3',
        # Performance académique
        'score_mean_w3', 'score_std_w3',
        # Comportement détaillé
        'active_days_cp_w3',
        'n_submitted_cp_w3',
        'n_late_cp_w3',
        'assessment_clicks_cp_w3',
        'content_clicks_cp_w3',
        # Statiques Jour 0
        'num_of_prev_attempts', 'studied_credits',
        'imd_score', 'highest_education_num',
        'age_band_num', 'registration_lead_time',
    ],
    7: [
        'clicks_sum_w7', 'clicks_mean_w7', 'clicks_std_w7',
        'score_mean_w7', 'score_std_w7',
        'active_days_cp_w7',
        'n_submitted_cp_w7',
        'n_late_cp_w7',
        'assessment_clicks_cp_w7',
        'content_clicks_cp_w7',
        'num_of_prev_attempts', 'studied_credits',
        'imd_score', 'highest_education_num',
        'age_band_num', 'registration_lead_time',
    ],
    12: [
        'clicks_sum_w12', 'clicks_mean_w12', 'clicks_std_w12',
        'score_mean_w12', 'score_std_w12',
        'active_days_cp_w12',
        'n_submitted_cp_w12',
        'n_late_cp_w12',
        'assessment_clicks_cp_w12',
        'content_clicks_cp_w12',
        'num_of_prev_attempts', 'studied_credits',
        'imd_score', 'highest_education_num',
        'age_band_num', 'registration_lead_time',
    ],
    38: [
        'total_clicks', 'mean_score',
        'weighted_score',
        'submission_rate',
        'n_active_days',
        'num_of_prev_attempts', 'studied_credits',
        'imd_score', 'highest_education_num', 'age_band_num',
    ],
}

In [ ]:
# ════════════════════════════════════════════════════════════
# COUCHE 2 — FILTRAGE HYBRIDE
# Profil + risk score + étudiants similaires
# ════════════════════════════════════════════════════════════

def couche2_filtrage_hybride(student_row, final_df_ref, embeddings_ref, semaine_courante):
    """
    Recommandation personnalisée basée sur :
    - Profil K-Means de l'étudiant
    - Score de risque GRU
    - Etudiants similaires qui ont réussi (filtrage collaboratif)
    """
    profile_col = PROFILE_COL_BY_CP[semaine_courante]
    profile = student_row.get( profile_col, student_row.get('profile_label', 'At-Risk Engager'))

    risk_score = student_row.get('risk_score', 0.5)
    sid        = student_row['id_student']
    reg        = student_row.get('click_regularity', 10)

    # ── Etape 1 : Difficulté selon risk score ─────────────────
    niveau = get_difficulty(risk_score)

    # ── Etape 2 : Contenu selon profil ────────────────────────
    content_config = CONTENT_BY_PROFILE.get(
        profile, CONTENT_BY_PROFILE['At-Risk Engager']
    )

    # description cohérente avec le niveau réel
    description_adaptee = get_difficulty_description(profile, niveau)


    # ── Etape 3 : Filtrage collaboratif ───────────────────────
    # A S3 : peu d'historique → utiliser uniquement la stratégie par profil
    if semaine_courante == 3:
      top_acts = content_config['content_types']

    else:
      # À S7/S12/S38 : profil + filtrage collaboratif
      feat_cols = [
          c for c in FEAT_COLS_BY_CP.get(semaine_courante, FEAT_COLS_BY_CP[38])
          if c in final_df_ref.columns
      ]

      student_vec = final_df_ref[
          final_df_ref['id_student'] == sid
      ][feat_cols].fillna(0).values

      peers_success_df = final_df_ref[
          (final_df_ref['at_risk'] == 0) &
          (final_df_ref['code_module'] == student_row['code_module']) &
          (final_df_ref['code_presentation'] == student_row['code_presentation'])
      ].copy()

      # ── Fallback si pas assez de pairs ────────────────────────
      MIN_PEERS = 10
      if len(peers_success_df) < MIN_PEERS:
        # Fallback 1 : même module, toutes présentations
        peers_success_df = final_df_ref[
            (final_df_ref['at_risk'] == 0) &
            (final_df_ref['code_module'] == student_row['code_module'])
        ].copy()

      if len(peers_success_df) < MIN_PEERS:
        # Fallback 2 : tous modules
        peers_success_df = final_df_ref[
            final_df_ref['at_risk'] == 0
        ].copy()


      peers_success = peers_success_df[
          feat_cols
      ].fillna(0).values

      if len(student_vec) > 0 and len(peers_success) > 0:
        similarities = cosine_similarity(
            student_vec,
            peers_success
        )[0]

        top5_idx = similarities.argsort()[-5:][::-1]
        top5_peers = peers_success_df.iloc[top5_idx]

        act_cols = [
            c for c in final_df_ref.columns
            if c.startswith('act_')
        ]

        if act_cols:
            top_acts = (
                top5_peers[act_cols]
                .mean()
                .nlargest(5)
                .index
            )
            top_acts = [
                a.replace('act_', '')
                for a in top_acts
            ]

            # Garder seulement les contenus pédagogiquement interprétables
            VALID_CONTENT_TYPES = [
                'resource',    # 2512 — document/fichier
                'oucontent',   # 615  — contenu interactif
                'url',         # 534  — ressource externe
                'forumng',     # 194  — social/collaboratif
                'quiz',        # 114  — évaluation formative
                'ouwiki',      # 43   — collaboratif avancé
            ]

            top_acts = [
                a for a in top_acts
                if a in VALID_CONTENT_TYPES
            ]

            # Si l'étudiant est Highly Engaged Learner et risk faible
            # → pas de quiz nécessaire
            if profile == 'Highly Engaged Learner' and niveau == 'difficile':
              top_acts = [a for a in top_acts if a != 'quiz']

            # Si le filtrage donne trop peu de contenus utiles
            if len(top_acts) < 3:
                top_acts = content_config['content_types']

        else:
            top_acts = content_config['content_types']

      else:
        top_acts = content_config['content_types']


    # ── Etape 4 : Rythme selon régularité ─────────────────────
    #rythme = get_rhythm(reg, risk_score=risk_score)

    return {
        'type'               : 'RECOMMANDATION_PERSONNALISEE',
        'priorite'           : ('haute' if risk_score >= 0.7
                                else 'moyenne'),
        'profil'             : profile,
        'niveau_difficulte'  : niveau,
        'difficulte'         : niveau,
        'content_types'      : top_acts[:3],
        'description_contenu': description_adaptee,   # ← corrigé
        #'rythme'             : rythme,
        'risk_score'         : round(risk_score, 3),
        'rationale'          : (content_config['rationale']
                                if niveau == 'difficile'
                                else (
                                    f"High risk score ({risk_score:.3f}) — "
                                    f"consolidation prioritized despite the profile {profile}"
                                    if profile in ['Highly Engaged Learner', 'Assessment Specialist', 'Mid-Engager']
                                    else
                                    f"High risk score ({risk_score:.3f}) — "
                                    f"consolidation needed for the profile {profile}"
                                    )
                                ),

        'source'             : 'hybride',
        'message'            : (
            f"Profile {profile} with risk score {risk_score:.3f}"
        ),
        'action'             : (
            f"Suggest {description_adaptee.lower()} "   # ← corrigé
            f"at a {niveau} level"
        ),
    }



In [ ]:
def get_trajectory_at_checkpoint(student_id, code_module, code_presentation, pcea_df, checkpoint):
    """
    Calcule la trajectoire uniquement à partir des données disponibles jusqu'au checkpoint courant.

    Checkpoint 3  → utilise uniquement profile_label_w3 (pas assez de points → Stable par défaut)
    Checkpoint 7  → utilise w3 et w7
    Checkpoint 12 → utilise  w7 et w12
    Final (38)    → utilise  w12 et profile_label
    """
    row = pcea_df[
        (pcea_df['id_student']        == student_id) &
        (pcea_df['code_module']        == code_module) &
        (pcea_df['code_presentation']  == code_presentation)
    ]

    if len(row) == 0:
        return 'Stable'   # fallback

    row = row.iloc[0]
    delta_threshold=1
    # Colonnes disponibles selon le checkpoint
    if checkpoint <= 3:
        # Un seul point → pas de tendance calculable
        return 'Beginning of journey'

    elif checkpoint <= 7:
        # w3 et w7 disponibles
        prev = row['risk_level_w3']
        curr = row['risk_level_w7']

    elif checkpoint <= 12:
        # w7, w12 disponibles
        prev = row['risk_level_w7']
        curr = row['risk_level_w12']

    else:
        prev = row['risk_level_w12']
        curr = row['risk_level']

    if pd.isna(prev) or pd.isna(curr):
        return 'Stable'

    # Recalculer la tendance
    delta = curr - prev

    if delta >= delta_threshold:
        return 'Decline'        # risk_level increased -> worse
    elif delta <= -delta_threshold:
        return 'Improvement'    # risk_level decreased -> better
    else:
        return 'Stable'

In [ ]:
# ════════════════════════════════════════════════════════════
# Couche 1 → si None → Couche 2
# ════════════════════════════════════════════════════════════

def generer_recommandation(student_row, weekly_data, final_df_ref, embeddings_ref, semaine_courante , pcea_df, assessments_df):

    """Pipeline M5 : Couche 1 → si None → Couche 2"""

    trajectory = get_trajectory_at_checkpoint(
        student_id        = student_row['id_student'],
        code_module       = student_row['code_module'],
        code_presentation = student_row['code_presentation'],
        pcea_df           = pcea_df,
        checkpoint        = semaine_courante
    )

    reco = couche1_regles_metier(
        student_row, weekly_data, semaine_courante, assessments_df
    )
    if reco is None:
        reco = couche2_filtrage_hybride(
            student_row, final_df_ref,
            embeddings_ref, semaine_courante
        )

    # Contexte commun
    reco['id_student']        = student_row['id_student']
    reco['code_module'] = student_row.get('code_module')
    reco['code_presentation'] = student_row.get('code_presentation')
    reco['semaine']           = semaine_courante

    profile_col = PROFILE_COL_BY_CP[semaine_courante]
    reco['profile_label'] = student_row.get(profile_col, student_row.get('profile_label', 'Inconnu'))

    reco['risk_score_predit'] = student_row.get('risk_score', None)
    reco['trajectory']        = trajectory
    return reco


In [ ]:
print("CP =", CP_PRINCIPAL)

print(
    "len(preds) =",
    len(checkpoint_results[CP_PRINCIPAL]['preds'])
)

print(
    "len(student_ids) =",
    len(student_ids)
)

In [ ]:
# ════════════════════════════════════════════════════════════
# APPLICATION SUR TOUS LES ÉTUDIANTS
# ════════════════════════════════════════════════════════════
print("\nGénération des recommandations...")
print("=" * 60)

final_with_profiles = final_df_wcdmacp.copy()
CP_PRINCIPAL = 3

risk_df = pd.DataFrame({
    'id_student'        : [student_ids[i][0]
                           for i in range(len(student_ids))],
    'code_module'       : [student_ids[i][1]
                           for i in range(len(student_ids))],
    'code_presentation' : [student_ids[i][2]
                           for i in range(len(student_ids))],
    'risk_score'        : checkpoint_results[CP_PRINCIPAL]['preds']
                          if len(checkpoint_results[CP_PRINCIPAL]['preds'])
                          == len(student_ids)
                          else np.zeros(len(student_ids)),
})

final_with_profiles = final_with_profiles.merge(
    risk_df, on=student_id_cols, how='left'
)

recommandations  = []
SEMAINE_COURANTE = CP_PRINCIPAL
print(f"Checkpoint : semaine {SEMAINE_COURANTE}")
print(f"Étudiants  : {len(final_with_profiles):,}")

for _, row in final_with_profiles.iterrows():
    reco = generer_recommandation(
        student_row      = row,
        weekly_data      = weekly_combined,
        final_df_ref     = final_with_profiles,
        embeddings_ref   = embeddings_full,
        semaine_courante = SEMAINE_COURANTE,
        pcea_df          = pcea_df,
        assessments_df   = assessments_df
    )
    recommandations.append(reco)

reco_df = pd.DataFrame(recommandations)

print(f"\nRecommandations générées : {len(reco_df):,}")
print(f"\nDistribution par type :")
print(reco_df['type'].value_counts().to_string())
print(f"\nDistribution par source :")
print(reco_df['source'].value_counts().to_string())
print(f"\nDistribution par priorité :")
print(reco_df['priorite'].value_counts().to_string())


In [ ]:
# ════════════════════════════════════════════════════════════
# ANALYSE DES RECOMMANDATIONS
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ANALYSE DES RECOMMANDATIONS")
print("=" * 60)

reco_with_truth = reco_df.merge(
    final_with_profiles[student_id_cols + ['at_risk']],
    on='id_student', how='left'
)

print("\nTaux at_risk par source :")
print(reco_with_truth.groupby('source')['at_risk']
      .mean().round(3).to_string())

print("\nTaux at_risk par type d'alerte :")
print(reco_with_truth.groupby('type')['at_risk']
      .mean().round(3).to_string())

rules_triggered = reco_df[
    reco_df['source'] == 'règles'
]['rule'].value_counts()

if len(rules_triggered) > 0:
    print(f"\nRègles déclenchées :")
    rule_names = {
        'R1': 'Inactivité totale',
        'R2': 'Décrochage prolongé',
        'R3': 'Silence pré-évaluation',
        'R4': 'Chute brutale ≥50%',
        'R5': 'Aucune soumission',
    }
    for rule, count in rules_triggered.items():
        print(f"  {rule} ({rule_names.get(rule, rule)}) : "
              f"{count:,} étudiants")

diff_by_profile = pd.crosstab(
    reco_df['profile_label'],
    reco_df['niveau_difficulte']
)
print(f"\nNiveau de difficulté par profil :")
display(diff_by_profile)


In [ ]:
# ════════════════════════════════════════════════════════════
# VISUALISATIONS
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("VISUALISATIONS M5")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Module M5 — Analyse des recommandations', fontweight='bold', fontsize=14)

# ── Plot 1 : Répartition couche 1 vs couche 2 ────────────────
source_counts = reco_df['source'].value_counts()
colors_source = ['#DC2626', '#2563EB']
axes[0,0].pie(source_counts.values,
              labels=source_counts.index,
              colors=colors_source,
              autopct='%1.1f%%',
              startangle=90,
              wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0,0].set_title('Répartition Couche 1 vs Couche 2',
                     fontweight='bold')

# ── Plot 2 : Types d'alertes ─────────────────────────────────
type_counts = reco_df['type'].value_counts()
bars = axes[0,1].barh(type_counts.index, type_counts.values,
                       color='#7C3AED', alpha=0.85)
for bar, val in zip(bars, type_counts.values):
    axes[0,1].text(bar.get_width() + 50,
                   bar.get_y() + bar.get_height()/2,
                   f'{val:,}', va='center', fontweight='bold')
axes[0,1].set_title("Types d'alertes et recommandations",
                     fontweight='bold')
axes[0,1].set_xlabel("Nombre d'étudiants")
axes[0,1].grid(alpha=0.3, axis='x')

# ── Plot 3 : Difficulté par profil ───────────────────────────
diff_by_profile.plot(kind='bar', ax=axes[1,0],
                      color=['#16A34A', '#D97706', '#DC2626'],
                      alpha=0.85)
axes[1,0].set_title('Niveau de difficulté par profil',
                     fontweight='bold')
axes[1,0].set_xlabel('Profil')
axes[1,0].set_ylabel("Nombre d'étudiants")
axes[1,0].tick_params(axis='x', rotation=15)
axes[1,0].legend(title='Difficulté')

# ── Plot 4 : Priorité par profil ─────────────────────────────
prio_by_profile = pd.crosstab(
    reco_df['profile_label'],
    reco_df['priorite']
)
prio_by_profile.plot(kind='bar', ax=axes[1,1],
                      color=['#16A34A', '#D97706', '#DC2626'],
                      alpha=0.85)
axes[1,1].set_title("Priorité d'intervention par profil",
                     fontweight='bold')
axes[1,1].set_xlabel('Profil')
axes[1,1].set_ylabel("Nombre d'étudiants")
axes[1,1].tick_params(axis='x', rotation=15)
axes[1,1].legend(title='Priorité')

plt.tight_layout()
plt.show()


In [ ]:
# ════════════════════════════════════════════════════════════
# EXEMPLES DE RECOMMANDATIONS
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("EXEMPLES DE RECOMMANDATIONS")
print("=" * 60)

def safe_value(v):
    # Cas liste / tableau
    if isinstance(v, (list, np.ndarray)):
      return v if len(v) > 0 else 'N/A'

    # Cas simple
    return v if pd.notna(v) else 'N/A'

def afficher_recommandation(reco):
    """Affiche une recommandation de façon lisible."""
    print(f"\n{'─'*55}")
    source_icon = '🔴' if reco['source'] == 'règles' else '🔵'
    print(f"{source_icon} SOURCE : {reco['source'].upper()}")
    print(f"   Étudiant     : {reco['id_student']}")
    print(f"   Semaine      : {reco['semaine']}")
    print(f"   Profil       : {reco['profile_label']}")
    print(f"   Trajectoire  : {reco.get('trajectory', 'N/A')}")
    if reco.get('risk_score_predit') is not None:
        rs   = reco['risk_score_predit']
        flag = '⚠️' if rs >= OPTIMAL_THRESHOLD else '✅'
        print(f"   Risk score   : {rs:.3f} {flag}")
    print(f"\n   TYPE         : {reco['type']}")
    print(f"   PRIORITÉ     : {reco['priorite'].upper()}")
    if 'message' in reco:
        print(f"   MESSAGE      : {reco['message']}")
    print(f"   ACTION       : {safe_value(reco.get('action'))}")
    print(f"   DIFFICULTÉ   : {safe_value(reco.get('niveau_difficulte', reco.get('difficulte')))}")
    print(f"   CONTENU      : {safe_value(reco.get('content_types'))}")
    #print(f"   RYTHME       : {safe_value(reco.get('rythme'))}")
    print(f"   JUSTIFICATION: {safe_value(reco.get('rationale'))}")


# CORRECTION 5 : choisir les exemples les plus représentatifs

# Cas 1 — Règles : ALERTE_CRITIQUE avec risk score le plus élevé
cas_r1 = reco_df[reco_df['rule'] == 'R1'] if 'rule' in reco_df.columns else pd.DataFrame()
if len(cas_r1) > 0:
    print("\n1. Cas Couche 1 — Règle R1 (inactivité totale) :")
    afficher_recommandation(
        cas_r1.loc[cas_r1['risk_score_predit'].fillna(0).idxmax()]
    )

cas_r4 = reco_df[reco_df['rule'] == 'R4'] if 'rule' in reco_df.columns else pd.DataFrame()
if len(cas_r4) > 0:
    print("\n2. Cas Couche 1 — Règle R4 (chute brutale) :")
    afficher_recommandation(
        cas_r4.loc[cas_r4['risk_score_predit'].fillna(0).idxmax()]
    )

# Cas 3 — Hybride avec risk élevé (≥ 0.7) — étudiant le plus à risque
cas_hybride_risk = reco_df[
    (reco_df['source'] == 'hybride') &
    (reco_df['risk_score_predit'].fillna(0) >= 0.7)
]
if len(cas_hybride_risk) > 0:
    print("\n3. Cas Couche 2 — Hybride (risk élevé) :")
    afficher_recommandation(
        cas_hybride_risk.loc[
            cas_hybride_risk['risk_score_predit'].idxmax()
        ]
    )

# Cas 4 — Hybride avec faible risque — étudiant le moins à risque
cas_hybride_ok = reco_df[
    (reco_df['source'] == 'hybride') &
    (reco_df['risk_score_predit'].fillna(1) < OPTIMAL_THRESHOLD)
]
if len(cas_hybride_ok) > 0:
    print("\n4. Cas Couche 2 — Hybride (faible risque) :")
    afficher_recommandation(
        cas_hybride_ok.loc[
            cas_hybride_ok['risk_score_predit'].idxmin()
        ]
    )



In [ ]:
# ════════════════════════════════════════════════════════════
# SAUVEGARDE
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("SAUVEGARDE M5")
print("=" * 60)

SAVE_PATH = OUTPUT_PATH + "recommandations/"
os.makedirs(SAVE_PATH, exist_ok=True)

reco_df.to_csv(
    SAVE_PATH + 'recommandations_w3_v4_m5.csv', index=False
)
print(f"recommandations_m5.csv sauvegardé")
print(f"   {len(reco_df):,} recommandations")
print(f"   {reco_df.columns.tolist()}")

In [ ]:
# ════════════════════════════════════════════════════════════
# RÉSUMÉ FINAL M5
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("RÉSUMÉ FINAL M5")
print("=" * 60)

n_regles  = (reco_df['source'] == 'règles').sum()
n_hybride = (reco_df['source'] == 'hybride').sum()
n_total   = len(reco_df)

print(f"\nTotal étudiants traités : {n_total:,}")
print(f"\nCouche 1 — Règles métier :")
print(f"  {n_regles:,} étudiants ({n_regles/n_total*100:.1f}%)")
if len(rules_triggered) > 0:
    for rule, count in rules_triggered.items():
        print(f"  → {rule} : {count:,} étudiants")

print(f"\nCouche 2 — Filtrage hybride :")
print(f"  {n_hybride:,} étudiants ({n_hybride/n_total*100:.1f}%)")
print(f"\nNiveaux de difficulté :")
print(reco_df['niveau_difficulte'].value_counts().to_string())
print(f"\nPriorités d'intervention :")
print(reco_df['priorite'].value_counts().to_string())

print(f"\nM5 terminé")

In [ ]:
import pandas as pd
SAVE_PATH = OUTPUT_PATH + "recommandations/"
# Charger le fichier
df = pd.read_csv(
  SAVE_PATH + "recommandations_w7_v5_m5.csv"
)


In [ ]:
import pandas as pd
SAVE_PATH = OUTPUT_PATH + "recommandations/"
# Charger le fichier
df_reco = pd.read_csv(
  SAVE_PATH + "recommandations_w7_v5_m5.csv"
)

In [ ]:
PROFILE_DESC = {
    'Resource Skimmer': 'Structured and guided content',
    'At-Risk Engager': 'Interactive content + support forums',
    'Mid-Engager': 'Performance and consolidation-oriented content',
    'Assessment Specialist': 'Assessment-oriented and in-depth content',
    'Highly Engaged Learner': 'Advanced and collaborative content'
}

In [ ]:
mask = df_reco['source'] == 'règles'

df_reco.loc[mask, 'description_contenu'] = (
    df_reco.loc[mask, 'profile_label']
           .map(PROFILE_DESC)
)

In [ ]:
df = pd.read_csv(
    SAVE_PATH + "recommandations_w7_v5_m5.csv"
)

df["description_contenu"] = df["description_contenu"].replace(
    "Contenu avancé et collaboratif",
    "Advanced and collaborative content"
)

df.to_csv(
    SAVE_PATH + "recommandations_w7_v5_m5.csv",
    index=False
)

In [ ]:
df_reco.to_csv(
    SAVE_PATH + 'recommandations_w7_v5_m5.csv',
    index=False
)

In [ ]:
def adjust_priority(base_priority, risk_score):

    priority_rank = {
        'moyenne': 1,
        'haute': 2,
        'critique': 3
    }

    if risk_score >= 0.80:
        score_priority = 'critique'
    elif risk_score >= 0.60:
        score_priority = 'haute'
    else:
        score_priority = 'moyenne'

    return max(
        base_priority,
        score_priority,
        key=lambda p: priority_rank[p]
    )

# Modifier uniquement les recommandations hybrides
mask = df['source'].str.lower() == 'hybride'

df.loc[mask, 'priorite'] = df.loc[mask].apply(
    lambda row: adjust_priority(
        row['priorite'],
        row['risk_score_predit']
    ),
    axis=1
)


In [ ]:
# Sauvegarder
df_reco.to_csv(
    SAVE_PATH + 'recommandations_w7_v5_m5.csv',
    index=False
)
